# Phase 4 - Count-direction steering (mitigation)
Phase 2c proved up_blocks.0 self-attention at early steps causally sets the count. Here we learn a **donor-free** 'more-objects' direction there (mean high-count - mean low-count activations) and ADD it during generation, sweeping the strength alpha. A monotonic rise of the rendered count vs alpha = a usable control knob.

This is NOT the project's prior failed steering (that was mid/late, cross-attn, correlational) - it steers the proven causal site.

**Runtime:** GPU (~20 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml, torch
import matplotlib.pyplot as plt
from src.prompts import build_prompt, generate_grid
from src.pipeline import (load_sdxl, generate, catalog_attention_sites,
                          select_probe_sites, generate_and_capture,
                          generate_with_steer)
from src.probes import count_direction
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase4.yaml')
raw = yaml.safe_load(open('configs/phase4.yaml'))
steer_steps, alphas = raw['steer_steps'], raw['alphas']
block, attn = raw['patch_block'], raw['patch_attn']
obj = cfg.objects[0]
pipe = load_sdxl(); det = Detector()
sites = [s for s in select_probe_sites(catalog_attention_sites(pipe.unet))
         if block in s and s.endswith(attn)]
def cnt(img):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
print('steer sites:', sites)

In [ ]:
# TRAIN: capture pooled early self-attn + rendered count across the grid.
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
feats = {st: {s: [] for s in sites} for st in steer_steps}
rendered = []
for i, p in enumerate(grid):
    img, snaps = generate_and_capture(pipe, p.text, p.seed, sites, steer_steps,
                                      cfg.num_inference_steps)  # default pool reducer
    rendered.append(cnt(img))
    for st in steer_steps:
        for s in sites:
            feats[st][s].append(snaps.get(st, {}).get(s))
    if (i + 1) % 10 == 0: print(f'{i+1}/{len(grid)}')
rendered = np.array(rendered)
print('captured', len(grid), 'train images; rendered range', rendered.min(), rendered.max())

In [ ]:
# DIRECTION: mean(high-count) - mean(low-count) per (step, site).
directions = {}
for st in steer_steps:
    directions[st] = {}
    for s in sites:
        X = np.array([v for v in feats[st][s] if v is not None])
        d = count_direction(X, rendered)
        directions[st][s] = torch.tensor(d, dtype=torch.float32)
print('built directions for', len(steer_steps), 'steps x', len(sites), 'sites')

In [ ]:
# DOSE-RESPONSE: sweep alpha on held-out eval prompts.
rows = []
for a in alphas:
    for cnt_req in raw['eval_counts']:
        prompt = build_prompt(cnt_req, obj)
        for seed in raw['eval_seeds']:
            img = generate_with_steer(pipe, prompt, seed, directions, float(a),
                                      steer_steps, cfg.num_inference_steps)
            rows.append({'alpha': a, 'req': cnt_req, 'seed': seed, 'rendered': cnt(img)})
    print('alpha', a, 'done')
dfd = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
dfd.to_csv('results/phase4_dose.csv', index=False)
dfd.groupby('alpha')['rendered'].mean()

In [ ]:
# Plot mean rendered count vs alpha (monotonic rise = a real count knob).
m = dfd.groupby('alpha')['rendered'].mean()
se = dfd.groupby('alpha')['rendered'].sem()
fig, ax = plt.subplots(figsize=(6, 5))
ax.errorbar(m.index, m.values, yerr=se.values, marker='o', capsize=3)
ax.set_xlabel('steering strength alpha'); ax.set_ylabel('mean rendered count')
ax.set_title('Dose-response: does the count direction control the count?')
plt.tight_layout()
plt.savefig('results/phase4_dose.png', dpi=100, bbox_inches='tight'); plt.show()

In [ ]:
# Eyeball: one eval prompt at alpha = min, 0, max.
seed0, req0 = raw['eval_seeds'][0], raw['eval_counts'][0]
prompt0 = build_prompt(req0, obj)
trio = [min(alphas), 0, max(alphas)]
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
for j, a in enumerate(trio):
    img = generate_with_steer(pipe, prompt0, seed0, directions, float(a),
                              steer_steps, cfg.num_inference_steps)
    axes[j].imshow(img); axes[j].axis('off')
    axes[j].set_title(f"'{prompt0}' | alpha={a} | count={cnt(img)}", fontsize=9)
plt.tight_layout()
plt.savefig('results/phase4_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **Mean rendered count rises monotonically with alpha** = the learned direction is a real, donor-free **count knob** -> we can steer the count at the causal site without a donor. Mitigation then = calibrate alpha per requested count (next step). And the eyeball should show coherently more animals as alpha grows.
- **Flat / non-monotonic** = a single pooled channel-direction isn't enough (the count needs spatial structure, which pooling discarded). Fallback: inject a per-count spatial TEMPLATE (averaged donor activations) instead of a pooled vector - the donor patch worked, so a template bank should too.
- Either outcome is informative: it tells us whether the count is a linear direction (steerable) or a spatial pattern (template-only) at the causal site.